In [1]:
%pip install pandas numpy matplotlib seaborn plotly altair ydata-profiling

In [2]:
# generator

from pathlib import Path
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
Path("freshmart_m2/data").mkdir(parents=True, exist_ok=True)

stores   = [f"S{i:03d}" for i in range(1, 21)]
products = [f"P{i:04d}" for i in range(1, 101)]
full = pd.date_range("2026-07-01", "2026-08-15", freq="D")

# store heterogeneity: some opened late (fewer active days), some low-traffic
open_offset = rng.integers(0, 30, len(stores))          # days after 1 Jul each store "opens"
traffic     = rng.uniform(0.4, 1.6, len(stores))        # relative volume multiplier
store_open  = dict(zip(stores, open_offset))
store_traf  = dict(zip(stores, traffic))

rows = []
tid = 0
for s in stores:
    active = full[store_open[s]:]                         # this store's active window
    n_s = int(1500 * store_traf[s])                       # its transaction count
    for _ in range(n_s):
        d = rng.choice(active)
        q = rng.poisson(3) + 1
        up = round(float(rng.uniform(20, 500)), 2)
        rows.append((f"T{tid:07d}", d, s, rng.choice(products), q, up, round(q*up,2)))
        tid += 1
df = pd.DataFrame(rows, columns=["transaction_id","transaction_date","store_id",
                                 "product_id","quantity","unit_price","revenue"])
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
# imperfections
df.loc[rng.choice(df.index, 250, replace=False), "quantity"] = np.nan
df = pd.concat([df, df.sample(150, random_state=7)], ignore_index=True)
df.to_csv("freshmart_m2/data/pos_transactions.csv", index=False)

pd.DataFrame({"store_id":stores,
    "city":rng.choice(["Pune","Mumbai","Bengaluru","Hyderabad"],len(stores)),
    "store_type":rng.choice(["Metro","Express","Neighbourhood"],len(stores))
}).to_csv("freshmart_m2/data/store_master.csv", index=False)
pd.DataFrame({"product_id":products,
    "category":rng.choice(["Dairy","Beverages","Snacks","Personal Care","Staples"],len(products)),
    "lead_time_days":rng.integers(1,8,len(products))
}).to_csv("freshmart_m2/data/product_master.csv", index=False)
print(df.shape, df.transaction_date.min().date(), df.transaction_date.max().date())


(31598, 7) 2026-07-03 2026-08-15


In [3]:
# Mission 1 - what actually arrived? Establish shape, grain, coverage, quality.
import pandas as pd

df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
print("shape:", df.shape)
print("date :", df["transaction_date"].min().date(), "to", df["transaction_date"].max().date())
print("stores:", df["store_id"].nunique(), "| products:", df["product_id"].nunique())
print("\nmissing values per column:")
print(df.isna().sum())
print("\nduplicate transaction_ids:", df["transaction_id"].duplicated().sum())
print("\nnumeric summary:")
print(df[["quantity", "unit_price", "revenue"]].describe().round(2))


shape: (31598, 7)
date : 2026-07-03 to 2026-08-15
stores: 20 | products: 100

missing values per column:
transaction_id        0
transaction_date      0
store_id              0
product_id            0
quantity            252
unit_price            0
revenue               0
dtype: int64

duplicate transaction_ids: 150

numeric summary:
       quantity  unit_price   revenue
count  31346.00    31598.00  31598.00
mean       4.00      260.54   1042.45
std        1.73      138.80    755.21
min        1.00       20.01     20.05
25%        3.00      140.06    443.10
50%        4.00      261.45    882.08
75%        5.00      381.58   1462.77
max       13.00      500.00   5963.16


In [4]:
# CHALLENGE 1 solution: classify each quality finding as ENGINEERING (contract) or LOCAL (analysis fix).
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])

findings = []
dup = df["transaction_id"].duplicated().sum()
findings.append(("duplicate transaction_ids", dup,
    "ENGINEERING", "uniqueness is a contract guarantee the pipeline must enforce upstream"))
missing_q = df["quantity"].isna().sum()
findings.append(("missing quantity", missing_q,
    "ENGINEERING", "completeness of a core measure belongs in the data contract, not each notebook"))
neg_rev = (df["revenue"] < 0).sum()
findings.append(("negative revenue", neg_rev,
    "LOCAL/none", "none found here; would be a contract range-check if present"))

print(f"{'finding':28} {'count':>7}  {'route':12} why")
for name,cnt,route,why in findings:
    print(f"{name:28} {cnt:>7}  {route:12} {why}")


finding                        count  route        why
duplicate transaction_ids        150  ENGINEERING  uniqueness is a contract guarantee the pipeline must enforce upstream
missing quantity                 252  ENGINEERING  completeness of a core measure belongs in the data contract, not each notebook
negative revenue                   0  LOCAL/none   none found here; would be a contract range-check if present


## Example

| Quality finding                                                 | Route              | Why?                                                                                  |
| --------------------------------------------------------------- | ------------------ | ------------------------------------------------------------------------------------- |
| `order_id` is missing                                           | **Engineering**    | Every downstream process needs a valid identifier.                                    |
| Duplicate `order_id` values                                     | **Engineering**    | Duplicates can systematically distort counts, joins and aggregations.                 |
| `order_date` contains invalid dates                             | **Engineering**    | Invalid dates can break time-based analysis across all consumers.                     |
| `quantity < 0` when negative quantity is not valid              | **Engineering**    | Violates a domain/business rule that should hold for the dataset.                     |
| `product_id` missing for 8% of records                          | **Engineering**    | Required field completeness is a reusable data-quality contract.                      |
| `region` has unexpected value `"Bengaluruu"`                    | **Engineering**    | Controlled categorical values should be standardized upstream.                        |
| Temperature has a few extreme values                            | **Local analysis** | Whether they are outliers depends on the analytical question and method.              |
| Sales are highly right-skewed                                   | **Local analysis** | Skewness is a property relevant to modelling/visualisation, not necessarily bad data. |
| Need to remove the top 1% of sales for a particular analysis    | **Local analysis** | The treatment depends on the analytical objective.                                    |
| Need to log-transform revenue                                   | **Local analysis** | Transformation depends on the statistical/modeling requirement.                       |
| Missing values in a variable used only for one model            | **Usually local**  | The appropriate imputation strategy depends on that model and analysis.               |
| Need to aggregate daily data into monthly data                  | **Local analysis** | This is an analytical transformation, not necessarily a data-quality defect.          |
| Column names inconsistent across files                          | **Engineering**    | Schema consistency should be guaranteed upstream.                                     |
| Units mixed between kg and tonnes                               | **Engineering**    | Unit consistency is a foundational data-contract issue.                               |
| A particular visualization needs records filtered to one region | **Local analysis** | Filtering is driven by the analytical question.                                       |


In [5]:
# Mission 2 - the automated second opinion. A broad profile as an accelerator.
import pandas as pd
from ydata_profiling import ProfileReport

df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
profile = ProfileReport(df, title="FreshMart POS Data Profile", minimal=True)
profile.to_file("freshmart_m2/profile.html")
print("profile written to freshmart_m2/profile.html - open it in a browser")


c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\samra\AppData\Local\Temp\ipykernel_214616\655043158.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 101.10it/s]

profile written to freshmart_m2/profile.html - open it in a browser


In [6]:
# CHALLENGE 2 solution: turn the profile into a triage table (valid / suspicious / invalid).
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
checks = []
checks.append(("transaction_id near-unique", df.transaction_id.nunique()/len(df),
    ">0.99 expected", "VALID - IDs should be unique; the small gap is the injected dups"))
checks.append(("quantity missing rate", df.quantity.isna().mean(),
    "should be ~0", "SUSPICIOUS - investigate the feed, do not silently impute"))
checks.append(("unit_price range", (df.unit_price.min(), df.unit_price.max()),
    "20-500 by design", "VALID - within the business range"))
for name,val,expect,verdict in checks:
    print(f"{name:26} value={val}  expect={expect}\n   -> {verdict}")


transaction_id near-unique value=0.9952528641053231  expect=>0.99 expected
   -> VALID - IDs should be unique; the small gap is the injected dups
quantity missing rate      value=0.007975188303057155  expect=should be ~0
   -> SUSPICIOUS - investigate the feed, do not silently impute
unit_price range           value=(20.01, 500.0)  expect=20-500 by design
   -> VALID - within the business range


### Example

| Finding                   | Evidence                    | Business meaning                            | Classification            | Action / Owner                      |
| ------------------------- | --------------------------- | ------------------------------------------- | ------------------------- | ----------------------------------- |
| Duplicate transaction IDs | 150 duplicates              | ID expected to be unique                    | **Contract**              | Pipeline check / Data Engineer      |
| Missing quantity          | 2.3%                        | Quantity is mandatory                       | **Contract**              | Pipeline validation / Data Engineer |
| Missing unit price        | 0.4%                        | Need to confirm whether price can be absent | **Needs domain decision** | Data Owner                          |
| Lead time = 30 days       | Observed in some categories | Acceptability depends on category           | **Needs investigation**   | Business/Data Owner                 |
| Invalid dates             | 0                           | No observed violation                       | **No action**             | —                                   |


In [7]:
# Mission 3 - which stores drive the growth? Compare on volume AND behaviour.
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])

store_day = (df.dropna(subset=["quantity"])
    .groupby(["store_id", "transaction_date"], as_index=False)
    .agg(units=("quantity", "sum"), revenue=("revenue", "sum"),
         transactions=("transaction_id", "nunique")))

store_summary = (store_day.groupby("store_id", as_index=False)
    .agg(units=("units", "sum"), revenue=("revenue", "sum"),
         active_days=("transaction_date", "nunique")))
store_summary["revenue_per_day"] = store_summary["revenue"] / store_summary["active_days"]

print(store_summary.sort_values("revenue", ascending=False).head(10).to_string(index=False))


In [ ]:
# CHALLENGE 3 solution: rank stability + where metric choice MATTERS.
# Finding: on uniform data the top store is stable, but the GAPS differ by metric
# and mid-table ranks reshuffle - so metric choice matters most for the "close" calls.
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
d = df.dropna(subset=["quantity"])
s = d.groupby("store_id").agg(revenue=("revenue","sum"), units=("quantity","sum"),
        active_days=("transaction_date","nunique")).reset_index()
s["rev_per_day"] = s.revenue/s.active_days

# rank each store on two metrics and show how many stores change rank position
r_rev = s.sort_values("revenue", ascending=False).reset_index(drop=True)
r_rpd = s.sort_values("rev_per_day", ascending=False).reset_index(drop=True)
pos_rev = {sid:i for i,sid in enumerate(r_rev.store_id)}
pos_rpd = {sid:i for i,sid in enumerate(r_rpd.store_id)}
moved = sum(1 for sid in s.store_id if pos_rev[sid] != pos_rpd[sid])
print(f"stores that change rank between revenue and rev_per_day: {moved} of {len(s)}")
print("top-1 is stable here, but mid-table reshuffles - metric choice matters for the close calls")
print("\ntop-5 by revenue vs by rev_per_day:")
print("revenue     :", list(r_rev.store_id.head(5)))
print("rev_per_day :", list(r_rpd.store_id.head(5)))


stores that change rank between revenue and rev_per_day: 14 of 20
top-1 is stable here, but mid-table reshuffles - metric choice matters for the close calls

top-5 by revenue vs by rev_per_day:
revenue     : ['S002', 'S013', 'S014', 'S009', 'S004']
rev_per_day : ['S002', 'S013', 'S014', 'S015', 'S012']


In [ ]:
# Mission 4 - the visualisation decision. Choose the chart, then the library.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])

# 1) a TIME question -> line
daily = (df.dropna(subset=["quantity"])
    .groupby("transaction_date", as_index=False)
    .agg(revenue=("revenue", "sum")))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(daily["transaction_date"], daily["revenue"])
ax.set_title("FreshMart Revenue Trend"); ax.set_xlabel("Date"); ax.set_ylabel("Revenue")
plt.tight_layout(); plt.savefig("freshmart_m2/trend.png", dpi=110)

# 2) a RANKING question -> bar (reuse store_summary from Mission 3)
store_summary = (df.dropna(subset=["quantity"])
    .groupby("store_id", as_index=False).agg(revenue=("revenue", "sum")))
top = store_summary.sort_values("revenue", ascending=False).head(10)
plt.figure(figsize=(7, 4)); sns.barplot(data=top, y="store_id", x="revenue")
plt.title("Top Stores by Revenue"); plt.tight_layout(); plt.savefig("freshmart_m2/top_stores.png", dpi=110)

# 3) a RELATIONSHIP question -> interactive scatter (Plotly)
fig = px.scatter(df.sample(min(4000, len(df)), random_state=42),
                 x="unit_price", y="quantity",color="store_id", title="Price vs Quantity")
fig.write_html("freshmart_m2/price_qty.html")
print("charts written: trend.png, top_stores.png, price_qty.html")


charts written: trend.png, top_stores.png, price_qty.html


In [ ]:
# CHALLENGE 4 solution: the store-ranking bar chart in Altair vs seaborn.
# Altair states the ENCODING declaratively (x/y/sort) - closer to "grammar of graphics".
import pandas as pd, altair as alt
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv")
top = (df.dropna(subset=["quantity"]).groupby("store_id", as_index=False)
         .agg(revenue=("revenue","sum")).sort_values("revenue", ascending=False).head(10))

chart = (alt.Chart(top).mark_bar().encode(
            x=alt.X("revenue:Q", title="Revenue"),
            y=alt.Y("store_id:N", sort="-x", title="Store"))
         .properties(title="Top Stores by Revenue (Altair)"))
chart.save("freshmart_m2/top_stores_altair.html")
print("Altair chart -> freshmart_m2/top_stores_altair.html")
print("Note: encoding is declared (x=revenue, y=store sorted by -x); the library figures out the rest.")


Altair chart -> freshmart_m2/top_stores_altair.html
Note: encoding is declared (x=revenue, y=store sorted by -x); the library figures out the rest.


In [ ]:
# Mission 5 - the store-cohort question. Group stores by their first active month.
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
stores_df = pd.read_csv("freshmart_m2/data/store_master.csv")

x = df.dropna(subset=["quantity"]).merge(stores_df, on="store_id").copy()
x["month"] = x["transaction_date"].dt.to_period("M").astype(str)
first_month = x.groupby("store_id")["transaction_date"].min().dt.to_period("M").astype(str)
x["cohort"] = x["store_id"].map(first_month)

cohort = (x.groupby(["cohort", "month"], as_index=False)
            .agg(units=("quantity", "sum"), revenue=("revenue", "sum")))
pivot = cohort.pivot(index="cohort", columns="month", values="revenue")
print(pivot.round(0).to_string())

month       2026-07     2026-08
cohort                         
2026-07  15316509.0  17375046.0


In [ ]:
# CHALLENGE 5 solution: a second cohort dimension (store_type) that aids replenishment.
import pandas as pd
df = pd.read_csv("freshmart_m2/data/pos_transactions.csv", parse_dates=["transaction_date"])
sm = pd.read_csv("freshmart_m2/data/store_master.csv")
d = df.dropna(subset=["quantity"]).merge(sm, on="store_id")
d["month"] = d.transaction_date.dt.to_period("M").astype(str)

# cohort by store_type (a replenishment-relevant segmentation, not just first-month)
pivot = d.pivot_table(index="store_type", columns="month", values="revenue",
                      aggfunc="sum").round(0)
print("revenue by store_type x month:")
print(pivot.to_string())
# why it helps replenishment: different formats have different velocity/lead-time needs
share = (d.groupby("store_type").revenue.sum()/d.revenue.sum()*100).round(1)
print("\nrevenue share by store_type (%):"); print(share.to_string())


revenue by store_type x month:
month            2026-07    2026-08
store_type                         
Express        5003800.0  5622776.0
Metro          6128730.0  8494228.0
Neighbourhood  4183980.0  3258042.0

revenue share by store_type (%):
store_type
Express          32.5
Metro            44.7
Neighbourhood    22.8


In [ ]:
!python --version

Python 3.13.9
